### Tools
LLMs can ask to use tools for tasks like getting data from APIs, searching online, querying databases, or executing code. A tool usually consists of two parts:

1. A definition (schema) that explains the tool — including its name, purpose, and expected inputs
2. The actual function or async function that runs when the tool is called

This helps the model interact with external systems and perform real-world actions. ([LangChain Docs][1])

[1]: https://docs.langchain.com/oss/python/langchain/tools?utm_source=chatgpt.com "Tools - Docs by LangChain"


In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

# api_key = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
# model = init_chat_model("groq:llama-3.1-8b-instant")
response = model.invoke("What is apple?")
response

AIMessage(content='<think>\nOkay, the user is asking "What is apple?" Let me start by understanding the possible interpretations.\n\nFirst, "apple" could refer to the fruit. That\'s the most straightforward. I should mention the botanical classification, like Malus domestica, and describe its characteristics: sweet, tart, used in cooking and baking. Maybe note the different varieties like Red Delicious or Granny Smith.\n\nThen there\'s Apple Inc., the tech company. They\'re known for products like iPhones, Macs, and the Apple ecosystem. Founded in 1976 by Steve Jobs, Steve Wozniak, and Ronald Wayne. Important to highlight their impact on technology and market presence. Also, mention their services like the App Store and iCloud.\n\nAlso, "apple" might be used in other contexts, like in phrases or idioms. For example, "the apple of my eye" or "an apple a day keeps the doctor away." But the user might not be looking for that, so maybe just a brief mention.\n\nNeed to check if the user is 

In [4]:
from langchain.tools import tool

@tool
def get_population(location:str)->str:
    """Get the current population in a given location"""
    populations = {
        "delhi": "33 million",
        "mumbai": "21 million",
        "new york": "8.5 million"
    }
    return populations.get(
        location.lower(),
        f"Population data for {location} not found."
    )


model_with_tools=model.bind_tools([get_population])

c:\Users\Administrator\Downloads\b150-genaiops\18_Langchain\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [5]:
response = model_with_tools.invoke("What's the population in new york ?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the population in New York. Let me check the tools available. There\'s a function called get_population that takes a location parameter. The required parameter is location, which should be a string. Since the user mentioned "New York," I need to pass that as the location. I should make sure to format the tool call correctly. The function doesn\'t require any other parameters, so the arguments should just include "location": "New York". Let me structure the JSON accordingly.\n', 'tool_calls': [{'id': 'yazbn8sn3', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_population'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 156, 'total_tokens': 283, 'completion_time': 0.197202918, 'completion_tokens_details': {'reasoning_tokens': 102}, 'prompt_time': 0.008971981, 'prompt_tokens_details': None, 'queue_time': 0.151936295, 'total_time': 0.20617

In [12]:
output=response.additional_kwargs['reasoning_content']
for i in output:
    if '.' in i:
        print('\n')
    print(i,end='')

Okay, the user is asking for the population in New York

. Let me check the tools available

. There's a function called get_population that takes a location parameter

. The required parameter is location, which should be a string

. Since the user mentioned "New York," I need to pass that as the location

. I should make sure to format the tool call correctly

. The function doesn't require any other parameters, so the arguments should just include "location": "New York"

. Let me structure the JSON accordingly

.


---

## Smart Restaurant Analogy 🍽️

### Step 1: Customer asks for something

```python
ai_msg = model_with_tools.invoke(messages)
```

Customer says:

> “Can I get a cold water bottle?”

Now the waiter (LLM) thinks:

* This does NOT require the kitchen
* Kitchen is for cooking meals
* Water bottle is already available in the fridge

So waiter decides:

> “Fridge is the correct place/tool.”

This is exactly how tool selection works in AI agents.

---

### Step 2: Correct tool performs the task

```python
tool_result = fridge.invoke(request)
```

The waiter gets the bottle from the fridge.

No unnecessary kitchen usage.

This represents:

* selecting the right tool
* avoiding unnecessary processing
* efficient execution

---

### Step 3: Final response to customer

```python
final_response = model_with_tools.invoke(messages)
```

Waiter returns:

> “Here is your cold water bottle.”

The LLM now formats the final human-friendly response.

---

### Mapping to AI Concepts 🤖

| Restaurant World            | AI World                 |
| --------------------------- | ------------------------ |
| Customer                    | User                     |
| Waiter                      | LLM                      |
| Fridge                      | Correct tool/API         |
| Kitchen                     | Another tool             |
| Waiter deciding where to go | Tool selection/reasoning |
| Delivering water            | Final response           |

---

### Core Idea

The important learning here is:

```text
LLM does not just call tools randomly.
It decides WHICH tool is most suitable for the task.
```

Example:

| User Request     | Correct Tool    |
| ---------------- | --------------- |
| Weather info     | Weather API     |
| Math calculation | Calculator tool |
| Database query   | SQL tool        |
| Web search       | Search API      |

This is the actual intelligence behind tool calling. ([langchain.com][1])

[1]: https://www.langchain.com/blog/tool-calling-with-langchain?utm_source=chatgpt.com "Tool Calling with LangChain"


### Tool Execution Loops

In [ ]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the population in new york ?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_population.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The population of New York is approximately 8.5 million.


In [ ]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call the function with location set to "Boston". Make sure the JSON is correctly formatted with the function name and arguments. No other functions are available, so this should be straightforward.\n', 'tool_calls': [{'id': '20cfda4p0', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 153, 'total_tokens': 262, 'completion_time': 0.199966855, 'completion_tokens_details': {'reasoning_tokens': 85}, 'prompt_time': 0.006519155, 'prompt_tokens_details': None, 'queue_time': 0.055720395, 'total_time': 0.20648601}, 'mod